# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [1]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

/Users/seneksa/STIC-B545/tac/.venv/lib/python3.13/site-packages/textblob/_text.py:218: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR1 = re.compile("^[A-Za-z]\.$")       # single letter, "T. De Smedt"
/Users/seneksa/STIC-B545/tac/.venv/lib/python3.13/site-packages/textblob/_text.py:219: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR2 = re.compile("^([A-Za-z]\.)+$")    # alternating letters, "U.S."
/Users/seneksa/STIC-B545/tac/.venv/lib/python3.13/site-packages/textblob_fr/_text.py:212: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR1 = re.compile("^[A-Za-z]\.$")       # single letter, "T. De Smedt"
/Users/seneksa/STIC-B545/tac/.venv/lib/python3.13/site-packages/textblob_fr/_text.py:213: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR2 = re.compile("^([A-Za-z]\.)+$")    # alternating letters, "U.S."


### Création d'une fonction `get_sentiment`

In [2]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [3]:
get_sentiment("Ce journal est vraiment super intéressant.")

This text is 65% positive and 75% subjective.


In [4]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

This text is 41% negative and 60% subjective.


## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [6]:
!uv pip install sentencepiece
!uv pip install transformers>=5.19.0
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

Using Python 3.13.15 environment at: /Users/seneksa/STIC-B545/tac/.venv
Checked 1 package in 1ms
zsh:1: 5.19.0 not found


/Users/seneksa/STIC-B545/tac/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1103 20:20:16.932000 40249 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### Chargement du modèle

In [7]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 48692.10it/s]


### Analyser le sentiment d'une phrase

In [8]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

[{'label': 'POSITIVE', 'score': 0.985571026802063}]

In [9]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")

[{'label': 'NEGATIVE', 'score': 0.9271842837333679}]